In [1]:
import torch
import os
import sys
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'code'))

from model_builder import YoloV1_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = YoloV1_model().to(device)

model.load_state_dict(torch.load('../model/Epoch-86.pth', map_location=device, weights_only=False)['model_state_dict'])

<All keys matched successfully>

In [2]:
import cv2
import numpy as np
import torch
def tensor_from_img(img):
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (448, 448))
    img_np_arr = img.astype(np.float32)/255.0
    img_np_arr = np.transpose(img_np_arr, (2, 0, 1))
    img_tensor = torch.from_numpy(img_np_arr).unsqueeze(0).to(device)
    return img_tensor

In [3]:
def match_predictions(iou_mat, prob_scores, iou_threshold=0.5):
    pred_num, gt_num = iou_mat.shape
    matched_gt = set()
    TP = 0
    FP = 0

    sorted_idx = np.argsort(prob_scores)[::-1]
    for pred_idx in sorted_idx:
        if gt_num == 0:
            FP += 1
            continue
        ious = iou_mat[pred_idx].numpy()

        candidates = [j for j in range(gt_num) if j not in matched_gt]

        best_gt = max(candidates, key=lambda j:ious[j])

        if ious[best_gt] >= iou_threshold:
            TP += 1
            matched_gt.add(best_gt)
        else:
            FP += 1
    FN = gt_num - len(matched_gt)
    return TP, FP, FN

In [4]:
CONF_THRESHOLD = 0.15
NMS_THRESHOLD = 0.30

S = 7
C = 1
B = 2

def get_target_from_output(model_output, conf_threshold, nms_threshold, Height, Width):
    boxes = []
    scores = []

    for row in range(S):
        for col in range(S):
            class_prob = float(model_output[row, col, 0])
            confidences = model_output[row, col, 1:3]

            for b in range(B):
                confidence = float(confidences[b])
                score = class_prob * confidence

                if not np.isfinite(score) or score < conf_threshold:
                    continue

                start = C + B + b * 4
                x, y, w, h = model_output[row, col, start:start+4]

                cx = (col + x) / S * Width
                cy = (row + y) / S * Height

                bw = w * Width
                bh = h * Height

                x1 = max(0, min(Width - 1, int(cx - bw / 2)))
                y1 = max(0, min(Height - 1, int(cy - bh / 2)))
                x2 = max(0, min(Width - 1, int(cx + bw / 2)))
                y2 = max(0, min(Height - 1, int(cy + bh / 2)))

                if x2 <= x1 or y2 <= y1:
                    continue

                boxes.append([x1, y1, x2-x1, y2-y1])
                scores.append(score)

    if not boxes:
        return [], []

    indices = cv2.dnn.NMSBoxes(
        boxes,
        scores,
        conf_threshold,
        nms_threshold
    )

    if len(indices) == 0:
        return [], []

    indices = np.array(indices).flatten()

    selected_boxes = [boxes[i] for i in indices]
    selected_scores = [scores[i] for i in indices]

    return selected_boxes, selected_scores

In [5]:
def load_ground_truth(label_path):
    boxes = []

    with open(label_path, 'r') as f:
        for line in f:
            values = list(map(float, line.split()))

            _, x, y, w, h = values

            boxes.append([x, y, w, h])

    return boxes

In [6]:
def calculate_iou(box_pascal, box_yolo, img_width=None, img_height=None):
    x_center, y_center, w, h = box_yolo
    
    # If YOLO is normalized [0, 1] and Pascal is in absolute pixels, scale YOLO to pixels
    if img_width is not None and img_height is not None:
        x_center *= img_width
        y_center *= img_height
        w *= img_width
        h *= img_height
        
    # Convert YOLO [x_center, y_center, w, h] to Pascal [xmin, ymin, xmax, ymax]
    yolo_xmin = x_center - (w / 2)
    yolo_ymin = y_center - (h / 2)
    yolo_xmax = x_center + (w / 2)
    yolo_ymax = y_center + (h / 2)
    
    box_yolo_pascal = [yolo_xmin, yolo_ymin, yolo_xmax, yolo_ymax]
    
    # Calculate intersection coordinates
    xmin_inter = max(box_pascal[0], box_yolo_pascal[0])
    ymin_inter = max(box_pascal[1], box_yolo_pascal[1])
    xmax_inter = min(box_pascal[2], box_yolo_pascal[2])
    ymax_inter = min(box_pascal[3], box_yolo_pascal[3])
    
    # Calculate intersection area
    inter_width = max(0.0, xmax_inter - xmin_inter)
    inter_height = max(0.0, ymax_inter - ymin_inter)
    inter_area = inter_width * inter_height
    
    # Calculate individual areas
    pascal_area = (box_pascal[2] - box_pascal[0]) * (box_pascal[3] - box_pascal[1])
    yolo_area = (box_yolo_pascal[2] - box_yolo_pascal[0]) * (box_yolo_pascal[3] - box_yolo_pascal[1])
    
    # Calculate union area
    union_area = pascal_area + yolo_area - inter_area
    
    if union_area <= 0.0:
        return 0.0
        
    return inter_area / union_area


In [7]:
def get_mat_from_predgt(pred_boxes, gt_boxes, height, width):
    mat = torch.zeros(len(pred_boxes), len(gt_boxes))
    for i, pred in enumerate(pred_boxes):
        px, py, pw, ph = pred

        pred_xyxy = [
            px,
            py,
            px + pw,
            py + ph
        ]
        for j, gt in enumerate(gt_boxes):
            mat[i, j] = calculate_iou(pred_xyxy, gt, width, height)
    return mat

In [8]:
from pathlib import Path
from tqdm import tqdm

image_path = Path('../data/TestingDataset/images/train')
label_path = Path('../data/TestingDataset/labels/train')
image_list = sorted(image_path.glob('*.png'))
model.eval()
total_TP = 0
total_FP = 0
total_FN = 0

for img_path in tqdm(image_list):
    img = cv2.imread(str(img_path))
    if img is None:
        raise FileNotFoundError(image_path)
    original = img.copy()
    H, W = img.shape[:2]
    img_tensor = tensor_from_img(img)
    with torch.inference_mode():
        predictions = model(img_tensor)
    predictions = predictions.reshape(1, S, S, C + B*5)
    pred = predictions[0].detach().cpu().numpy()

    pred_boxes, score_boxes = get_target_from_output(
        pred,
        CONF_THRESHOLD,
        NMS_THRESHOLD,
        H,
        W
    )
    gt_boxes = load_ground_truth(label_path/f"{img_path.stem}.txt")
    iou_mat = get_mat_from_predgt(pred_boxes, gt_boxes, H, W)
    TP, FP, FN = match_predictions(iou_mat, score_boxes, 0.5)
    total_TP += TP
    total_FN += FN
    total_FP += FP



precision = total_TP / (total_TP + total_FP + 1e-8)

recall = total_TP / (total_TP + total_FN + 1e-8)

f1 = 2 * precision * recall / (precision + recall + 1e-8)

print(f"True Positives: {total_TP}")
print(f"False Positives: {total_FP}")
print(f"False Negatives: {total_FN}")

print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

100%|██████████| 1154/1154 [01:23<00:00, 13.76it/s]

True Positives: 1140
False Positives: 2
False Negatives: 284
Precision: 0.9982
Recall: 0.8006
F1 Score: 0.8885
